In [1]:

import yfinance as yf
import pandas as pd
import numpy as np

# Downloading E-mini S&P 500 futures
es = yf.download(
    "ES=F",
    start="2025-07-30",
    end="2026-03-10",
    interval="60m",
)

es.columns = es.columns.droplevel(1)

es = es.reset_index()
es['Datetime'] = pd.to_datetime(es['Datetime'])
es = es.set_index('Datetime')

overnight = es[
    (es.index.time >= pd.to_datetime("18:00").time()) |
    (es.index.time <= pd.to_datetime("09:29").time())
].copy()

overnight['Return'] = overnight['Close'].pct_change()
overnight['Date'] = pd.to_datetime(overnight.index.date)
# shift evening trades to the next day
overnight.loc[overnight.index.time >= pd.to_datetime("18:00").time(), 'Date'] += pd.Timedelta(days=1)

overnight_features = overnight.groupby('Date').agg(
    Overnight_Return = ('Return', 'sum'),
    Overnight_Volatility = ('Return', 'std'),
    Futures_Last_Price = ('Close', 'last')
)
overnight_features = overnight_features.fillna(0)

overnight_features.to_csv("../data/Overnight_SP500_data.csv")
print(overnight_features.head())

[*********************100%***********************]  1 of 1 completed

            Overnight_Return  Overnight_Volatility  Futures_Last_Price
Date                                                                  
2025-07-30         -0.000506              0.000354             6412.25
2025-07-31          0.006948              0.002083             6456.75
2025-08-01         -0.022168              0.002764             6314.75
2025-08-02         -0.007952              0.005147             6264.50
2025-08-04          0.006290              0.000658             6304.00
